# CorrDiff Vietnam — Preprocessing Pipeline (tp_coarse → ERA5-Land tp)

Streamlined version: **single input variable** (`tp_coarse`) → **single output** (`tp`).  
All 12 other atmospheric inputs removed to reduce distraction in model training.

Produces:
- `PROCESSED_INPUT/era5_input_tp_{year}.nc` — yearly ERA5 coarse TP interpolated to fine grid
- `PROCESSED_OUTPUT/era5_land_tp_vietnam_{year}.nc` — yearly ERA5-Land target TP *(already done)*
- `ZARR_PATH/vietnam_data.zarr` — consolidated zarr (all years, both variables)
- `ZARR_PATH/stats.json` — normalization stats (train-only)

**Split:** 2017–2023 train | 2024 val | 2025 test

## 0. Environment

In [ ]:
!conda install -c conda-forge netcdf4 h5netcdf zarr -y

## 1. Config & shared helpers

In [ ]:
import gc
import json
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path
from scipy.interpolate import RegularGridInterpolator

# ── Paths ─────────────────────────────────────────────────────────────────────
# Raw ERA5 TP files (separate directory used during download)
RAW_INPUT_TP    = Path("/mnt/data/khaiht/data/vietnam/input_tp")
# Raw ERA5-Land monthly output files
RAW_OUTPUT_DIR  = Path("/mnt/data/khaiht/data/vietnam/output")

# Processed yearly input files  ← will be written by Section 2
PROCESSED_INPUT  = Path("/mnt/data/khaiht/data/vietnamvip_processed/input_tp_only")
# Processed yearly output files ← already exist
PROCESSED_OUTPUT = Path("/mnt/data/khaiht/data/vietnamvip_processed/output")

# Final zarr + stats
ZARR_DIR   = Path("/mnt/data/khaiht/data/vietnam_train_tp_only")
ZARR_PATH  = ZARR_DIR / "vietnam_data.zarr"
STATS_PATH = ZARR_DIR / "stats.json"

PROCESSED_INPUT.mkdir(parents=True, exist_ok=True)
ZARR_DIR.mkdir(parents=True, exist_ok=True)

# ── Domain ────────────────────────────────────────────────────────────────────
LAT_MIN, LAT_MAX = 5.8, 25.0
LON_MIN, LON_MAX = 102.0, 118.0

# ── Variables ─────────────────────────────────────────────────────────────────
# ERA5 coarse TP is the ONLY input channel — file: era5_sl_tp_{year}.nc
INPUT_TP_VAR = "tp"
OUTPUT_VAR   = "tp"   # ERA5-Land hourly TP, log1p-transformed

# ── Split years ───────────────────────────────────────────────────────────────
ALL_YEARS   = list(range(2017, 2026))
TRAIN_YEARS = list(range(2017, 2024))
VAL_YEARS   = [2024]
TEST_YEARS  = [2025]

print("Config ready.")
print(f"  PROCESSED_INPUT  : {PROCESSED_INPUT}")
print(f"  PROCESSED_OUTPUT : {PROCESSED_OUTPUT}")
print(f"  ZARR_PATH        : {ZARR_PATH}")
print(f"  STATS_PATH       : {STATS_PATH}")
print(f"  Train years: {TRAIN_YEARS}")
print(f"  Val   years: {VAL_YEARS}")
print(f"  Test  years: {TEST_YEARS}")

In [ ]:
# ── Shared helper functions ────────────────────────────────────────────────────

def clean_ds(ds):
    """Rename valid_time→time, drop expver/number dims, clear attrs."""
    if "valid_time" in ds.dims or "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    for dim in ("expver", "number"):
        if dim in ds.dims:
            ds = ds.isel({dim: 0}, drop=True)
        if dim in ds.coords:
            ds = ds.drop_vars(dim)
    ds.attrs = {}
    for v in ds.variables:
        ds[v].attrs = {}
    return ds


def crop(ds):
    """Crop to Vietnam bounding box. ERA5 is N→S so slice(MAX,MIN)."""
    return ds.sel(
        latitude=slice(LAT_MAX, LAT_MIN),
        longitude=slice(LON_MIN, LON_MAX),
    )


def get_land_mask():
    """
    Static land mask (True=land) derived from first processed ERA5-Land file.
    Shape: (n_lat_fine, n_lon_fine).
    """
    ref_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{ALL_YEARS[0]}.nc"
    ds = xr.open_dataset(ref_file)
    mask = (~ds[OUTPUT_VAR].isel(time=0).isnull()).values.astype(bool)
    fine_lat = ds["latitude"].values
    fine_lon = ds["longitude"].values
    ds.close()
    return mask, fine_lat, fine_lon


LAND_MASK, FINE_LAT, FINE_LON = get_land_mask()
print(f"Land mask loaded: {LAND_MASK.shape}  "
      f"({LAND_MASK.sum():,} land px / {LAND_MASK.size:,} total)")
print(f"Fine grid lat: {FINE_LAT.min():.2f} → {FINE_LAT.max():.2f}  "
      f"({len(FINE_LAT)} pts)")
print(f"Fine grid lon: {FINE_LON.min():.2f} → {FINE_LON.max():.2f}  "
      f"({len(FINE_LON)} pts)")

## 2. Process ERA5 coarse TP → `PROCESSED_INPUT/era5_input_tp_{year}.nc`

For each year we:
1. Load ERA5 single-level hourly TP (already true hourly, no differencing needed)
2. Convert m → mm, clip float32 noise, apply **log1p** (same transform as the target)
3. Bilinearly interpolate to the ERA5-Land fine grid using `scipy`
4. Apply the ERA5-Land land mask: ocean pixels → `NaN` (consistent with all other inputs)
5. Save as a single-variable `xr.Dataset` (`tp_coarse`), float32

**Why land-mask the coarse input?**  
Masking ensures ocean pixels carry `NaN` rather than interpolated open-ocean values.
The dataset class fills all input `NaN`s with 0.0 uniformly before passing the tensor
to the model; the loss function additionally ignores ocean pixels via the same mask.
This approach is identical to the one used in `corrdiff_interpolate_updated_v5`
where the ERA5 land mask is applied to the coarse grid via nearest-neighbour resampling
before any statistics or comparisons are computed.

**No normalization here.** Raw log1p(mm/hr) values are stored; z-score stats are
computed in Section 5 (train data only).

In [ ]:
def bilinear_to_fine(da_coarse, fill_ocean_nan=True):
    """
    Bilinearly interpolate a coarse ERA5 DataArray (lat/lon/time) to the
    ERA5-Land fine grid using scipy RegularGridInterpolator.

    Parameters
    ----------
    da_coarse     : xr.DataArray   shape (time, lat, lon)
    fill_ocean_nan: bool
        If True, ocean pixels (LAND_MASK=False) are set to NaN.
        Always True for tp_coarse — loss masks ocean anyway, and
        consistent NaN treatment simplifies nanmean stats later.

    Returns
    -------
    xr.DataArray   shape (time, n_lat_fine, n_lon_fine), float32
    """
    c_lat = da_coarse["latitude"].values
    c_lon = da_coarse["longitude"].values
    times = da_coarse["time"].values

    # RegularGridInterpolator needs ascending axes
    lat_asc  = c_lat[::-1] if c_lat[0] > c_lat[-1] else c_lat
    flip_lat = (c_lat[0] > c_lat[-1])

    # Fine-grid query points (ascending lat)
    fine_lat_asc = FINE_LAT[::-1] if FINE_LAT[0] > FINE_LAT[-1] else FINE_LAT
    pts = np.array(
        np.meshgrid(fine_lat_asc, FINE_LON, indexing="ij")
    ).reshape(2, -1).T   # (H*W, 2)

    out_arr = np.empty(
        (len(times), len(FINE_LAT), len(FINE_LON)), dtype=np.float32
    )

    for t_idx in range(len(times)):
        vals = da_coarse.values[t_idx]
        if flip_lat:
            vals = vals[::-1]

        vals = np.nan_to_num(vals, nan=0.0)

        interp = RegularGridInterpolator(
            (lat_asc, c_lon), vals,
            method="linear",
            bounds_error=False,
            fill_value=0.0,
        )
        out_fine = interp(pts).reshape(len(fine_lat_asc), len(FINE_LON))

        # Restore descending latitude order to match FINE_LAT
        if FINE_LAT[0] > FINE_LAT[-1]:
            out_fine = out_fine[::-1]

        # Clip any float32 interpolation noise below zero
        out_fine = np.clip(out_fine, 0.0, None)

        if fill_ocean_nan:
            out_fine = np.where(LAND_MASK, out_fine, np.nan)

        out_arr[t_idx] = out_fine.astype(np.float32)

    return xr.DataArray(
        out_arr,
        dims=["time", "latitude", "longitude"],
        coords={"time": times, "latitude": FINE_LAT, "longitude": FINE_LON},
        name=da_coarse.name,
    )


print("bilinear_to_fine() defined.")

In [ ]:
def era5_coarse_tp_to_log1p_mm(tp_da):
    """
    Convert ERA5 single-level hourly TP (metres) to log1p(mm/hr).

    ERA5 single-level TP from CDS is already true hourly accumulated —
    each timestep holds only the precipitation in that one hour.
    No differencing or reset handling is required.

    We apply log1p for the same reason it is applied to the ERA5-Land target:
      1. The distribution is zero-inflated with a heavy tail; log1p compresses
         the tail so z-score normalization is meaningful.
      2. After log1p + z-score, tp_coarse and the target tp live in the same
         numerical space, making it easier for the model to learn the residual.

    Parameters
    ----------
    tp_da : xr.DataArray  (time, lat, lon), units = metres

    Returns
    -------
    xr.DataArray  (time, lat, lon), units = log1p(mm/hr), float32
    """
    tp_mm  = (tp_da * 1000.0).clip(min=0.0)   # m → mm, clip float32 noise
    tp_log = np.log1p(tp_mm)                   # log1p to match target space
    return tp_log.astype("float32")


print("era5_coarse_tp_to_log1p_mm() defined.")

In [ ]:
# ── Main loop: process ERA5 coarse TP year by year ───────────────────────────
#
# File layout expected in RAW_INPUT_TP:
#   era5_sl_tp_{year}.nc      (ERA5 single-level hourly TP, one file per year)
#
# Output:
#   PROCESSED_INPUT/era5_input_tp_{year}.nc  — single variable 'tp_coarse'

for year in ALL_YEARS:
    out_file = PROCESSED_INPUT / f"era5_input_tp_{year}.nc"
    if out_file.exists():
        print(f"[SKIP] {out_file.name} already exists")
        continue

    print(f"\n{'='*60}")
    print(f"  Processing ERA5 coarse TP — {year}")
    print(f"{'='*60}")

    fp_tp = RAW_INPUT_TP / f"era5_sl_tp_{year}.nc"
    if not fp_tp.exists():
        raise FileNotFoundError(f"Missing: {fp_tp}")

    print(f"  Loading from {fp_tp.name}...", end=" ", flush=True)
    ds_raw = xr.open_dataset(fp_tp)
    ds_raw = clean_ds(ds_raw)
    ds_raw = crop(ds_raw)

    tp_log = era5_coarse_tp_to_log1p_mm(ds_raw[INPUT_TP_VAR])
    tp_log.name = "tp_coarse"
    ds_raw.close()
    print(f"shape={dict(tp_log.sizes)}  interp...", end=" ", flush=True)

    # Interpolate to fine grid; apply land mask (ocean → NaN)
    da_fine = bilinear_to_fine(tp_log, fill_ocean_nan=True)
    print("done")

    # Build output dataset & save
    ds_out = xr.Dataset({"tp_coarse": da_fine}).astype("float32")

    encoding = {
        "tp_coarse": {
            "zlib": True, "complevel": 1, "dtype": "float32",
            "chunksizes": (24, len(FINE_LAT), len(FINE_LON)),
        }
    }
    ds_out.to_netcdf(out_file, encoding=encoding)
    print(f"  ✅ Saved {out_file.name}  "
          f"[{ds_out.sizes['time']} timesteps]")

    del ds_out, da_fine, tp_log
    gc.collect()

print("\n✅ All ERA5 coarse TP input files processed.")

In [ ]:
# ── Quick sanity check on one processed coarse-TP input file ─────────────────
year_check = 2020
ds_check = xr.open_dataset(PROCESSED_INPUT / f"era5_input_tp_{year_check}.nc")
print(ds_check)
print("\nVariable info:")
da = ds_check["tp_coarse"]
nan_frac = float(np.isnan(da.values[0]).mean())
land_frac = float((~np.isnan(da.values[0])).mean())
print(f"  tp_coarse  shape={tuple(da.shape)}  dtype={da.dtype}")
print(f"  NaN frac (t=0): {nan_frac:.4f} (ocean)  |  Land frac: {land_frac:.4f}")
print(f"  min={float(np.nanmin(da.values)):.4f}  max={float(np.nanmax(da.values)):.4f}  "
      f"(log1p mm/hr space)")
ds_check.close()

## 3. Target dataset — ERA5-Land TP (already processed)

The yearly files in `PROCESSED_OUTPUT/era5_land_tp_vietnam_{year}.nc` were already produced.
Each file contains the `tp` variable in **log1p(mm/hr)** space, with ocean pixels as `NaN`.

This section validates them.

In [ ]:
print("Validating processed ERA5-Land TP files...")
print(f"{'Year':>6}  {'Steps':>6}  {'First ts':<20}  {'Last ts':<20}  {'Max(mm/hr)':>10}  {'NaN frac':>9}")
print("-" * 80)

for year in ALL_YEARS:
    fp = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    if not fp.exists():
        print(f"  {year}: ⚠️  FILE MISSING — run the target-processing notebook first")
        continue

    ds = xr.open_dataset(fp)
    times   = ds["time"].values
    raw_log = ds["tp"].values.astype(np.float32)
    ds.close()

    exp_len = 8784 if year in (2020, 2024) else 8760
    ok_len  = "✅" if len(times) == exp_len else f"❌ (got {len(times)})"

    max_mm   = float(np.nanmax(np.expm1(raw_log.astype(np.float64))))
    nan_frac = float(np.isnan(raw_log[0]).mean())

    print(f"  {year}  {len(times):>6}  {str(times[0])[:19]:<20}  "
          f"{str(times[-1])[:19]:<20}  {max_mm:>10.2f}  {nan_frac:>9.4f}  {ok_len}")

## 4. Consolidate all years into a single Zarr store

**One zarr for all years (train + val + test).**  
The `VietnamDataset` class handles the train/val/test split by filtering on time indices.

Layout inside the zarr:
- Variable: `tp_coarse` (ERA5 coarse TP on fine grid, log1p mm/hr, NaN on ocean)
- Variable: `tp` (ERA5-Land target, log1p mm/hr, NaN on ocean)
- Dims: `(time, latitude, longitude)`
- Chunks: `(1, n_lat, n_lon)` — one full spatial slice per chunk for fast random-access during training

In [ ]:
import zarr

total_timesteps = 0

for i, year in enumerate(ALL_YEARS):
    fp_in  = PROCESSED_INPUT  / f"era5_input_tp_{year}.nc"
    fp_out = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"

    if not fp_in.exists():
        raise FileNotFoundError(f"Missing processed coarse TP input: {fp_in}")
    if not fp_out.exists():
        raise FileNotFoundError(f"Missing processed ERA5-Land output: {fp_out}")

    ds_in  = xr.open_dataset(fp_in,  chunks=None)
    ds_out = xr.open_dataset(fp_out, chunks=None)

    # Align timestamps
    common_times = np.intersect1d(ds_in["time"].values, ds_out["time"].values)
    ds_in  = ds_in.sel(time=common_times)
    ds_out = ds_out.sel(time=common_times)

    ds_year = xr.merge([ds_in, ds_out[["tp"]]])

    n_lat = ds_year.sizes["latitude"]
    n_lon = ds_year.sizes["longitude"]
    n_t   = ds_year.sizes["time"]

    ds_year = ds_year.chunk({"time": 1, "latitude": -1, "longitude": -1})

    encoding = {
        v: {
            "compressor": None,
            "dtype":      "float32",
            "chunks":     (1, n_lat, n_lon),
        }
        for v in ds_year.data_vars
    }

    if i == 0:
        print(f"Creating zarr store at {ZARR_PATH} ...")
        ds_year.to_zarr(ZARR_PATH, mode="w", encoding=encoding, consolidated=True)
    else:
        ds_year.to_zarr(ZARR_PATH, mode="a", append_dim="time", consolidated=True)

    total_timesteps += n_t
    print(f"  {year}: {n_t} timesteps appended  (cumulative: {total_timesteps:,})")

    ds_in.close(); ds_out.close(); ds_year.close()
    del ds_in, ds_out, ds_year
    gc.collect()

zarr.consolidate_metadata(str(ZARR_PATH))
print(f"\n✅ Zarr written: {ZARR_PATH}")
print(f"   Total timesteps: {total_timesteps:,}")

In [ ]:
# ── Sanity-check the zarr store ───────────────────────────────────────────────
ds_check = xr.open_zarr(ZARR_PATH, consolidated=True)
print("Zarr store contents:")
print(f"  sizes     : {dict(ds_check.sizes)}")
print(f"  variables : {list(ds_check.data_vars)}")
print(f"  time[0]   : {ds_check.time.values[0]}")
print(f"  time[-1]  : {ds_check.time.values[-1]}")

sample = ds_check.isel(time=0).load()
for v in ds_check.data_vars:
    arr = sample[v].values
    n_nan  = int(np.isnan(arr).sum())
    n_land = int((~np.isnan(arr)).sum())
    print(f"  {v:<14}  land_pixels={n_land:,}  nan_pixels={n_nan:,}  "
          f"min={np.nanmin(arr):.4f}  max={np.nanmax(arr):.4f}")

ds_check.close()
print("\n✅ Zarr sanity check passed.")

## 5. Compute normalization statistics — **train data only** (2017–2023)

Per-variable `mean` and `std` computed **exclusively from training timesteps**
to prevent data leakage.

| Variable | Transform stored in zarr | Normalization |
|---|---|---|
| ERA5 coarse TP (`tp_coarse`) | **log1p(mm/hr)** | z-score: `(x - mean) / std` |
| ERA5-Land target TP (`tp`) | log1p(mm/hr) | z-score: `(x - mean) / std` over **land pixels only** |

Ocean pixels are `NaN` for both variables → `nanmean` / `nanstd` automatically gives
land-only statistics.

In [ ]:
import json

print(f"Computing normalization stats over training years: {TRAIN_YEARS}")

ds_train = xr.open_zarr(ZARR_PATH, consolidated=True)
train_mask = np.isin(ds_train["time"].dt.year.values, TRAIN_YEARS)
ds_train   = ds_train.isel(time=train_mask)
print(f"Training timesteps: {ds_train.sizes['time']:,}")

stats = {"input": {}, "output": {}}

# ── tp_coarse (input) ──────────────────────────────────────────────────────
print("  tp_coarse...", end=" ", flush=True)
arr = ds_train["tp_coarse"].values.astype(np.float64)
mean_v = float(np.nanmean(arr))
std_v  = float(np.nanstd(arr))
std_v  = max(std_v, 1e-6)
stats["input"]["tp_coarse"] = {"mean": mean_v, "std": std_v}
print(f"mean={mean_v:.4f}  std={std_v:.4f}")
del arr; gc.collect()

# ── tp target (output) ────────────────────────────────────────────────────
print("  tp (target)...", end=" ", flush=True)
arr_tp  = ds_train["tp"].values.astype(np.float64)
mean_tp = float(np.nanmean(arr_tp))
std_tp  = float(np.nanstd(arr_tp))
std_tp  = max(std_tp, 1e-6)
stats["output"]["tp"] = {"mean": mean_tp, "std": std_tp}
print(f"mean={mean_tp:.4f}  std={std_tp:.4f}")
del arr_tp; gc.collect()

ds_train.close()

with open(STATS_PATH, "w") as f:
    json.dump(stats, f, indent=2)

print(f"\n✅ Saved stats to {STATS_PATH}")
print(json.dumps(stats, indent=2))

## 6. Final verification

In [ ]:
# ── Verify stats.json ─────────────────────────────────────────────────────────
with open(STATS_PATH) as f:
    loaded_stats = json.load(f)

print("stats.json contents:")
print(f"  {'Variable':<14}  {'Mean':>12}  {'Std':>12}  {'Group'}")
print("  " + "-"*52)
for v, d in loaded_stats["input"].items():
    print(f"  {v:<14}  {d['mean']:>12.5f}  {d['std']:>12.5f}  input")
for v, d in loaded_stats["output"].items():
    print(f"  {v:<14}  {d['mean']:>12.5f}  {d['std']:>12.5f}  output")

In [ ]:
# ── Check zarr split coverage ─────────────────────────────────────────────────
ds_z = xr.open_zarr(ZARR_PATH, consolidated=True)
years_in_zarr = np.unique(ds_z["time"].dt.year.values)
print(f"Years in zarr: {sorted(years_in_zarr.tolist())}")

for split, yrs in [("TRAIN", TRAIN_YEARS), ("VAL", VAL_YEARS), ("TEST", TEST_YEARS)]:
    mask = np.isin(ds_z["time"].dt.year.values, yrs)
    n    = int(mask.sum())
    print(f"  {split:5s} ({yrs[0]}–{yrs[-1]}): {n:,} timesteps")

ds_z.close()
print("\n✅ Zarr integrity verified.")

In [ ]:
# ── Normalisation round-trip spot-check ───────────────────────────────────────
with open(STATS_PATH) as f:
    st = json.load(f)

ds_z = xr.open_zarr(ZARR_PATH, consolidated=True)
sample_t = 100

print("Round-trip normalisation check (timestep index 100):")
for v in ["tp_coarse", "tp"]:
    grp  = "output" if v == "tp" else "input"
    mean = st[grp][v]["mean"]
    std  = st[grp][v]["std"]

    raw   = ds_z[v].isel(time=sample_t).values.astype(np.float64)
    normd = (raw - mean) / std
    recon = normd * std + mean

    err = np.nanmax(np.abs(recon - raw))
    print(f"  {v:<14}  max_abs_err={err:.2e}  {'✅' if err < 1e-5 else '⚠️'}")

ds_z.close()

In [ ]:
# ── Final file inventory ──────────────────────────────────────────────────────
import subprocess

print("Processed INPUT (coarse TP) files:")
for fp in sorted(PROCESSED_INPUT.glob("era5_input_tp_*.nc")):
    size_mb = fp.stat().st_size / 1e6
    print(f"  {fp.name}  {size_mb:.0f} MB")

print("\nProcessed OUTPUT (ERA5-Land) files:")
for fp in sorted(PROCESSED_OUTPUT.glob("era5_land_tp_vietnam_*.nc")):
    size_mb = fp.stat().st_size / 1e6
    print(f"  {fp.name}  {size_mb:.0f} MB")

print(f"\nZarr store: {ZARR_PATH}")
result = subprocess.run(["du", "-sh", str(ZARR_PATH)], capture_output=True, text=True)
print(f"  Size: {result.stdout.strip()}")

print(f"\nStats file: {STATS_PATH}")
print(f"  Size: {STATS_PATH.stat().st_size} bytes")

## 7. Data Analysis & Visualizations

**Standalone cells.** Every code cell below imports everything it needs from
`viz_utils.py` (a small module shipped alongside this notebook) instead of
relying on variables set by earlier cells. Restart the kernel and run any
single cell in this section on its own — it will still work. Shared logic
(map styling, colormap, land-mask/coarse-TP loaders, the peak-star toggle)
lives in `viz_utils.py` in one place, so a change there is picked up by
every figure the next time its cell runs.

**Peak-star toggle.** Set `SHOW_PEAK_STAR = False` at the top of
`viz_utils.py` (or `import viz_utils; viz_utils.SHOW_PEAK_STAR = False`
before calling a plotting cell) to drop the ★ peak-pixel marker from every
extreme-event map, snapshot, animation frame, and the top-4 panel below.

**Colormap decision — one scale for all intensity maps.** The previous
version used `YlOrRd` for the multi-year max map and `Blues` for the p99
map, even though both panels encode the same physical quantity (mm/hr).
Mixing hue families across panels of the same variable makes them read as
different metrics and makes it harder to compare directly. `viz_utils.py`
now defines a blue precipitation scale, `PRECIP_CMAP`/`PRECIP_NORM`
(continuous matplotlib `Blues` on a fixed log scale), used everywhere rainfall intensity is
plotted in this notebook. This keeps "blue = light rain" — the intuitive
read you preferred from the p99 map — while extreme values still escalate
to red, which is the convention reviewers of an extreme-precipitation paper
will expect for spotting the dangerous tail. It's also the exact palette
already used for the headline results figures in `vietnam_results.ipynb`,
so figures across both notebooks now look like they belong to the same
paper. If you'd rather use a perceptually-uniform continuous map instead
that's a reasonable alternative — the banded scale is better for calling
out discrete intensity thresholds (which matters for a hazard/extremes
paper), a continuous map is better if you want smooth gradients for
model-error diagnostics elsewhere in the pipeline.

**East Sea vs. South China Sea.** The maps below label the sea east of
Vietnam using the same styling/position as `vietnam_results.ipynb`. A few
facts to inform the choice for an Elsevier submission:
- The IHO (International Hydrographic Organization) standard/internationally
  registered name for this body of water is **"South China Sea."** That's
  the name most non-Vietnamese readers, reviewers, and indexing databases
  will recognize by default.
- **"East Sea" (Biển Đông)** is Vietnam's national usage, and "Bien Dong" /
  "East Sea" already appears in a meaningful number of Vietnamese-authored
  papers in international journals (including some Elsevier titles),
  usually without controversy — but it is not the IHO-standard name, and a
  reviewer or reader unfamiliar with the regional context may briefly
  wonder what body of water is meant, or read it as a stance in the
  territorial dispute (this sea also has other locally-used names, e.g. the
  Philippines' "West Philippine Sea").
- Recommendation for an international, geopolitically-neutral submission:
  label it **"South China Sea"** as the primary/only label on the figures
  themselves (safest default, matches IHO nomenclature, avoids the
  appearance of taking a position), and if you want to retain the local
  context, do it in prose/caption rather than on the map itself — e.g.
  *"...the South China Sea (known in Vietnam as Biển Đông / the East
  Sea)..."* the first time it's mentioned in the text. A one-line
  disclaimer is a reasonable, low-cost way to preempt reviewer questions,
  e.g. in a caption footnote or acknowledgments: *"Geographic feature names
  on figures follow standard IHO nomenclature; their use does not reflect
  a position on regional territorial claims."*
- This is ultimately an editorial/author-team call, not something with a
  single objectively "correct" answer — I'm giving you the practical
  publishing tradeoffs, not a claim about the underlying territorial
  dispute itself.
- To act on either choice, just change `SEA_LABEL` in `viz_utils.py`
  (currently `"East Sea"`) — every map picks it up automatically. If you
  drop the label entirely, set `SEA_LABEL = None` and skip calling
  `add_east_sea_label` (or leave it — an empty label is a no-op) — actually
  simplest is to set `SEA_LABEL = ""`.

### 7.0 Study Area Map (Vietnam)


In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.0  STUDY AREA MAP — Vietnam terrain & regions of interest  (standalone)
#      No rainfall data. Geographic reference figure for the "Study area"
#      subsection: coastal plain, Annamite (Truong Son) Range, Red River
#      Delta, Mekong Delta. Region shapes are hand-drawn/illustrative, not
#      administrative GIS boundaries. Uses the shared FULL_EXTENT and
#      island-label helpers, so Hoang Sa/Truong Sa and gridlines match every
#      other map in this notebook.
# ══════════════════════════════════════════════════════════════
from viz_utils import *
import matplotlib.patches as mpatches

proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(8, 10.5))
ax = fig.add_subplot(1, 1, 1, projection=proj)
ax.set_extent(list(FULL_EXTENT), crs=proj)

# Ocean / land base colors — kept local to this figure (the rainfall maps
# elsewhere use LAND_10M's plain cream fill; this one is purely illustrative)
ax.set_facecolor("#cfe8f3")
land_terrain = cfeature.NaturalEarthFeature(
    "physical", "land", "10m", facecolor="#eef3e1", edgecolor="none", zorder=0)
ax.add_feature(land_terrain)
ax.add_feature(COAST_10M)
ax.add_feature(BORDERS_10M)
add_gridlines(ax)

def region_patch(coords, color, alpha=0.45, z=5):
    ax.add_patch(mpatches.Polygon(coords, closed=True, transform=proj,
                 facecolor=color, edgecolor=color, alpha=alpha, linewidth=1.3, zorder=z))

# Annamite (Truong Son) Range — interior spine, north to south
region_patch([
    (105.0, 19.6), (106.3, 19.2), (107.0, 17.6), (107.6, 15.8),
    (108.1, 14.0), (108.4, 12.2), (108.1, 10.6), (107.2, 11.1),
    (106.6, 12.6), (106.0, 14.5), (105.4, 16.5), (104.8, 18.3),
], "#a9744f")

# Red River Delta (north)
region_patch([
    (105.3, 21.6), (106.5, 21.5), (106.9, 20.7), (106.3, 19.9),
    (105.4, 20.0), (104.9, 20.8),
], "#2f855a")

# Mekong Delta (south)
region_patch([
    (104.8, 10.8), (106.8, 10.6), (106.7, 9.0), (105.6, 8.6),
    (104.6, 9.4), (104.5, 10.2),
], "#2f855a")

# Coastal plain — thin strip hugging the central coastline
region_patch([
    (105.9, 18.6), (106.6, 18.2), (108.6, 16.4), (109.3, 14.9),
    (109.4, 12.4), (108.6, 10.9), (108.0, 11.6), (107.7, 13.6),
    (108.0, 15.6), (107.4, 17.3), (106.4, 18.1),
], "#e8c468")

add_map_labels(ax, transform=proj)

legend_handles = [
    mpatches.Patch(facecolor="#a9744f", edgecolor="#a9744f", alpha=0.45,
                   label="Annamite (Trường Sơn) Range"),
    mpatches.Patch(facecolor="#2f855a", edgecolor="#2f855a", alpha=0.45,
                   label="River deltas (Red River / Mekong)"),
    mpatches.Patch(facecolor="#e8c468", edgecolor="#e8c468", alpha=0.45,
                   label="Coastal plain"),
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.savefig(str(PROCESSED_OUTPUT / "study_area_map.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Study area map saved.")


**Figure — Study area.** The domain spans roughly 6–25°N, 102–118°E, covering Vietnam and its immediate surroundings. Highlighted schematically: the Annamite (Trường Sơn) Range running the length of the interior; the Red River Delta (north) and Mekong Delta (south); and the coastal plain between the range and the South China Sea. Hòang Sa (Paracel) and Trường Sa (Spratly) Islands are marked. Region shapes are illustrative, not administrative GIS boundaries.


### 7.1 Annual Statistics Table

In [ ]:
import os, sys, time, warnings, json, glob, re, struct

PROJECT_ROOT  = "/home/khaiht/oggy_climate"
CORRDIFF_ROOT = "/home/khaiht/oggy_climate/physicsnemo/examples/generative/corrdiff"
for p in [PROJECT_ROOT, CORRDIFF_ROOT]:
    if p not in sys.path: sys.path.insert(0, p)

from viz_utils import *

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.1  ANNUAL STATISTICS TABLE  (standalone)
# ══════════════════════════════════════════════════════════════
from viz_utils import *

print("Computing annual statistics (ERA5-Land 0.1°)...")
df_annual = compute_annual_stats()
print(df_annual.to_string(index=False))


**Colormap note.** The dataset intensity maps use `PRECIP_CMAP` (matplotlib 
`Blues`) on a fixed logarithmic scale, `PRECIP_NORM = LogNorm(0.1, 150)`, so every 
panel shares one colorbar range and the blue precipitation maps stay directly 
comparable and visually distinct from the yellow-red inference figures in 
`vietnam_results.ipynb`. No extra packages are required.

### 7.2 Monthly Climatology — Fine vs Coarse

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.2  MONTHLY CLIMATOLOGY  (standalone)
# ══════════════════════════════════════════════════════════════
from viz_utils import *

print("Loading data and computing monthly climatology...")
tp_fine_mm = load_fine_tp()
tp_coarse_mm, _, _ = load_coarse_tp_masked()

months      = list(range(1, 13))
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fine_monthly_mean, fine_monthly_max     = [], []
coarse_monthly_mean, coarse_monthly_max = [], []

for m in months:
    tp_m = tp_fine_mm.sel(time=tp_fine_mm.time.dt.month == m)
    fine_monthly_mean.append(float(tp_m.mean()))
    fine_monthly_max.append(float(tp_m.max()))

    tp_m_c = tp_coarse_mm.sel(time=tp_coarse_mm.time.dt.month == m)
    coarse_monthly_mean.append(float(tp_m_c.mean()))
    coarse_monthly_max.append(float(tp_m_c.max()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x, w = np.arange(12), 0.35
axes[0].bar(x-w/2, fine_monthly_mean,   w, label='ERA5-Land 0.1°', color='#2196F3', alpha=0.85)
axes[0].bar(x+w/2, coarse_monthly_mean, w, label='ERA5 0.25°',     color='#FF5722', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(month_names)
axes[0].set_ylabel('Mean rainfall (mm/hr)')
axes[0].set_title('Mean', fontsize=11)
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x-w/2, fine_monthly_max,   w, label='ERA5-Land 0.1°', color='#2196F3', alpha=0.85)
axes[1].bar(x+w/2, coarse_monthly_max, w, label='ERA5 0.25°',     color='#FF5722', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(month_names)
axes[1].set_ylabel('Max hourly rainfall (mm/hr)')
axes[1].set_title('Max', fontsize=11)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(pad=1.5)
plt.savefig(str(PROCESSED_OUTPUT / "stat_monthly_climatology.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Monthly climatology saved.")


**Figure — Monthly climatology of hourly rainfall over Vietnam (2017–2025), ERA5-Land 0.1° vs. land-masked ERA5 0.25°.** Left: monthly mean; right: monthly maximum hourly rainfall. ERA5-Land captures higher extremes than ERA5 in every month, consistent with its finer grid resolving sub-grid precipitation peaks.


### 7.3 Diurnal Cycle

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.3  DIURNAL CYCLE (Vietnam, all years, local time UTC+7)  (standalone)
# ══════════════════════════════════════════════════════════════
from viz_utils import *

print("Loading data and computing diurnal cycle...")
tp_fine_mm = load_fine_tp()
tp_coarse_mm, _, _ = load_coarse_tp_masked()

UTC_OFFSET = 7
fine_diurnal, coarse_diurnal = [], []
for h in range(24):
    fine_diurnal.append(float(tp_fine_mm.sel(time=tp_fine_mm.time.dt.hour == h).mean()))
    coarse_diurnal.append(float(tp_coarse_mm.sel(time=tp_coarse_mm.time.dt.hour == h).mean()))

local_hours    = [(h + UTC_OFFSET) % 24 for h in range(24)]
order          = np.argsort(local_hours)
fine_d_local   = [fine_diurnal[i]   for i in order]
coarse_d_local = [coarse_diurnal[i] for i in order]
x_local        = sorted(local_hours)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_local, fine_d_local,   'o-',  color='#1565C0', lw=2, ms=5, label='ERA5-Land 0.1°')
ax.plot(x_local, coarse_d_local, 's--', color='#BF360C', lw=2, ms=5, label='ERA5 0.25° (land-masked)')
ax.set_xlabel('Local time (UTC+7)')
ax.set_ylabel('Mean hourly rainfall (mm/hr)')
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], rotation=45)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(PROCESSED_OUTPUT / "stat_diurnal_cycle.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Diurnal cycle saved.")


**Figure — Diurnal cycle of mean hourly rainfall over Vietnam, 2017–2025 (local time, UTC+7).** ERA5-Land 0.1° (solid) vs. land-masked ERA5 0.25° (dashed).


### 7.4 Inter-annual Variability

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.4  INTER-ANNUAL VARIABILITY — annual domain maximum  (standalone)
# ══════════════════════════════════════════════════════════════
from viz_utils import *

print("Loading data and computing inter-annual variability...")
tp_coarse_mm, _, _ = load_coarse_tp_masked()

ann_max_fine, ann_max_coarse = [], []
for year in ALL_YEARS:
    fn   = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_y = xr.open_dataset(fn)
    tp_f = np.expm1(ds_y["tp"])
    ann_max_fine.append(float(tp_f.max()))

    tp_c_y = tp_coarse_mm.sel(time=tp_coarse_mm.time.dt.year == year)
    ann_max_coarse.append(float(tp_c_y.max()))
    ds_y.close()

ratios_max    = [f / max(c, 0.01) for f, c in zip(ann_max_fine, ann_max_coarse)]
ratio_max_str = f"{min(ratios_max):.1f}–{max(ratios_max):.1f}×"

fig, ax = plt.subplots(figsize=(10, 5))
ax.axvspan(2017-0.5, 2023+0.5, alpha=0.07, color='blue',  label='Train')
ax.axvspan(2023+0.5, 2024+0.5, alpha=0.07, color='green', label='Val')
ax.axvspan(2024+0.5, 2025+0.5, alpha=0.07, color='red',   label='Test')
ax.set_xticks(ALL_YEARS); ax.tick_params(axis='x', rotation=45)
ax.grid(alpha=0.3)

ax.plot(ALL_YEARS, ann_max_fine,   'o-',  color='#1565C0', lw=2, ms=7, label='ERA5-Land 0.1°')
ax.plot(ALL_YEARS, ann_max_coarse, 's--', color='#BF360C', lw=2, ms=7, label='ERA5 0.25° (land-masked)')
ax.set_ylabel('Max hourly rainfall (mm/hr)')
ax.legend(fontsize=9)

plt.tight_layout(pad=1.5)
plt.savefig(str(PROCESSED_OUTPUT / "stat_interannual.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Inter-annual variability saved.")


**Figure — Annual domain-maximum hourly rainfall, 2017–2025, ERA5-Land 0.1° vs. land-masked ERA5 0.25°.** Shaded bands mark the train/val/test split. ERA5-Land exceeds ERA5 in every year (see printed `ratio_max_str` output above for the exact multiplier range).


### 7.5 Spatial Maps of Extreme Rainfall

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.5  SPATIAL MAPS — Multi-year max & p99  (standalone)
#      Figure A: ERA5 0.25° (left) | ERA5-Land 0.1° (right)
#      All-Blues; max panels share one scale, p99 panels share another (data-driven vmax).
# ══════════════════════════════════════════════════════════════
from viz_utils import *
import matplotlib.patches as mpatches

print("Computing spatial statistics maps...")
tp_coarse_mm, fine_land_da, fine_land_coarse = load_coarse_tp_masked()

tp_fine_max = tp_fine_p99 = None
for year in ALL_YEARS:
    fn   = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_y = xr.open_dataset(fn)
    tp   = np.expm1(ds_y["tp"])
    if tp_fine_max is None:
        tp_fine_max = tp.max(dim="time").compute()
        tp_fine_p99 = tp.quantile(0.99, dim="time").compute()
    else:
        tp_fine_max = xr.concat([tp_fine_max, tp.max(dim="time").compute()],
                                dim="year").max(dim="year")
        tp_fine_p99 = xr.concat([tp_fine_p99, tp.quantile(0.99, dim="time").compute()],
                                dim="year").max(dim="year")
    ds_y.close()

tp_coarse_max = tp_coarse_mm.max(dim="time").compute()
tp_coarse_p99 = tp_coarse_mm.quantile(0.99, dim="time").compute()

proj = ccrs.PlateCarree()

def _plot_map(ax, data, title, vmax):
    la = data.latitude.values; lo = data.longitude.values
    if la[0] > la[-1]:
        data = data.isel(latitude=slice(None, None, -1)); la = data.latitude.values
    im = ax.pcolormesh(lo, la, data.values, cmap=PRECIP_CMAP,
                        norm=precip_norm(vmax, scale="linear"),
                        transform=proj, shading="auto")
    ax.add_feature(BORDERS_10M); ax.add_feature(COAST_10M)
    ax.set_extent(list(FULL_EXTENT), crs=proj)
    add_gridlines(ax)
    plt.colorbar(im, ax=ax, orientation="vertical", fraction=0.03, pad=0.04, label="mm/hr")
    add_map_labels(ax, transform=proj)
    ax.set_title(title, fontsize=10, pad=6)
    return im

# shared vmax per quantity: max panels comparable to each other, p99 panels to each other
vmax_max = max(float(tp_coarse_max.max()), float(tp_fine_max.max()))
vmax_p99 = max(float(tp_coarse_p99.max()), float(tp_fine_p99.max()))

fig, axes = plt.subplots(2, 2, figsize=(14, 14),
                          subplot_kw={"projection": proj},
                          gridspec_kw={"wspace": 0.22, "hspace": 0.18})
_plot_map(axes[0,0], tp_coarse_max, "ERA5 0.25° — Max", vmax_max)
_plot_map(axes[0,1], tp_fine_max,   "ERA5-Land 0.1° — Max", vmax_max)
_plot_map(axes[1,0], tp_coarse_p99, "ERA5 0.25° — p99", vmax_p99)
_plot_map(axes[1,1], tp_fine_p99,   "ERA5-Land 0.1° — p99", vmax_p99)
plt.tight_layout(pad=1.5)
plt.savefig(str(PROCESSED_OUTPUT / "stat_spatial_extremes.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Spatial maps saved.")

**Figure — Spatial distribution of extreme rainfall over Vietnam, 2017–2025.** Left column: land-masked ERA5 0.25°; right column: ERA5-Land 0.1°. Top row: multi-year maximum hourly rainfall; bottom row: 99th-percentile hourly rainfall. Max and p99 rows are each on their own shared colour scale.


### 7.6 Top-10 Extreme Hourly Events

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.6  TOP-10 EXTREME HOURLY EVENTS (domain-wide)  (standalone)
# Fine/coarse ratio reflects spatial sharpening (sub-grid structure
# concentrated into finer pixels), not invisible sub-grid events;
# both grids see the same large-scale storm systems.
# ══════════════════════════════════════════════════════════════
from viz_utils import *

print("Finding top extreme hourly events per year...")
df_extreme = find_extreme_events()
print("\nTop extreme hourly rainfall events (ERA5-Land 0.1° vs ERA5 0.25° land-masked):")
print(df_extreme.to_string(index=False))

ratios      = df_extreme["Ratio fine/coarse-max"]
coarse_pts  = df_extreme["Coarse @ pt (mm/hr)"]
ratio_range = f"{ratios.min():.1f}\u2013{ratios.max():.1f}\u00d7"
pt_range    = f"{coarse_pts.min():.0f}\u2013{coarse_pts.max():.0f} mm/hr"

print(f"\nKey: ERA5-Land exceeds ERA5 domain-max by {ratio_range} (spatial sharpening).")
print(f"     Coarse at peak pixel = {pt_range} \u2014 both grids see the same storm systems.")


### 7.7 Extreme Event Snapshot — Fine vs Coarse

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.7  SNAPSHOT — Best extreme event  (standalone)
#      Figure A: ERA5 0.25° (left) | ERA5-Land 0.1° (right)
#      Uses the shared PRECIP_CMAP and the SHOW_PEAK_STAR toggle from viz_utils.
# ══════════════════════════════════════════════════════════════
from viz_utils import *
import matplotlib.patches as mpatches

df_extreme = find_extreme_events()
tp_coarse_mm, fine_land_da, fine_land_coarse = load_coarse_tp_masked()

best        = df_extreme.iloc[0]
target_time = pd.Timestamp(best["Time (UTC)"])
target_lat  = best["Fine lat"]
target_lon  = best["Fine lon"]
target_year = int(best["Year"])

print(f"Visualising: {target_time}  |  Fine peak: {best['Fine max (mm/hr)']} mm/hr")

fn = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{target_year}.nc"
ds_y = xr.open_dataset(fn)
tp_fine_snap   = np.expm1(ds_y["tp"].sel(time=target_time, method="nearest")).compute()
ds_y.close()
tp_coarse_snap = tp_coarse_mm.sel(time=target_time, method="nearest").compute()

fine_max       = float(tp_fine_snap.max())
coarse_max     = float(tp_coarse_snap.max())
ratio          = fine_max / max(coarse_max, 0.01)
coarse_at_peak = float(tp_coarse_snap.sel(
    latitude=target_lat, longitude=target_lon, method="nearest").values)

proj = ccrs.PlateCarree()

def _snap_map(ax, data, title):
    la = data.latitude.values; lo = data.longitude.values
    if la[0] > la[-1]:
        data = data.isel(latitude=slice(None,None,-1)); la = data.latitude.values
    im = ax.pcolormesh(lo, la, data.values, cmap=PRECIP_CMAP, norm=PRECIP_NORM,
                       transform=proj, shading="auto")
    ax.add_feature(BORDERS_10M); ax.add_feature(COAST_10M); ax.add_feature(LAND_10M)
    ax.set_extent(list(FULL_EXTENT), crs=proj)
    gl = add_gridlines(ax)
    if gl is not None:
        gl.xlabel_style = {"size": 8}; gl.ylabel_style = {"size": 8}
    plot_peak_star(ax, target_lon, target_lat, transform=proj)
    add_map_labels(ax, transform=proj)
    ax.set_title(title, fontsize=12, pad=7)
    return im

fig, axes = plt.subplots(1, 2, figsize=(15, 7),
                          subplot_kw={"projection": proj},
                          gridspec_kw={"wspace": 0.18})
_snap_map(axes[0], tp_coarse_snap, "ERA5 0.25°")
im1 = _snap_map(axes[1], tp_fine_snap, "ERA5-Land 0.1°")
cbar = fig.colorbar(im1, ax=axes, orientation="vertical",
                    fraction=0.025, pad=0.03, shrink=0.85, extend="both")
cbar.set_label("Hourly rainfall intensity (mm/hr)", fontsize=10)
cbar.ax.tick_params(labelsize=8)
plt.savefig(str(PROCESSED_OUTPUT / "stat_extreme_snapshot.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 Extreme snapshot saved.")

**Figure — Extreme rainfall snapshot, land-masked ERA5 0.25° (left) vs. ERA5-Land 0.1° (right).** Star marks the domain peak pixel. See the printed output above for the event timestamp and exact fine/coarse maxima and ratio.


### 7.8 Animation — 24-hour Evolution of an Extreme Event

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.8  ANIMATION — 24-hour evolution of an extreme event (GIF)  (standalone)
#      Ported from corrdiff_interpolate_updated_v5.ipynb Section 9.
#      Layout: side-by-side (coarse left, fine right).
#      Uses the shared PRECIP_CMAP and the SHOW_PEAK_STAR toggle from viz_utils.
# ══════════════════════════════════════════════════════════════
from viz_utils import *
import matplotlib.animation as animation
import matplotlib as mpl

df_extreme = find_extreme_events()
tp_coarse_mm, _, _ = load_coarse_tp_masked()

best      = df_extreme.iloc[0]
anim_year = int(best["Year"])
t_peak    = pd.Timestamp(best["Time (UTC)"])
t_start   = t_peak - pd.Timedelta(hours=12)
t_end     = t_peak + pd.Timedelta(hours=11)
print(f"Building animation {t_start} -> {t_end}  ({anim_year})")

fn = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{anim_year}.nc"
ds_y = xr.open_dataset(fn)
tp_fine_anim = np.expm1(
    ds_y["tp"].sel(time=slice(str(t_start), str(t_end)))
).compute()
ds_y.close()

tp_coarse_anim = tp_coarse_mm.sel(
    time=slice(str(t_start), str(t_end))
).sel(
    latitude=slice(LAT_MAX + 2, LAT_MIN - 2),
    longitude=slice(LON_MIN - 2, LON_MAX + 2)
).compute()

n_frames = tp_fine_anim.sizes["time"]

proj = ccrs.PlateCarree()
fig  = plt.figure(figsize=(14, 7))
gs   = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1, 1, 0.04], wspace=0.08)
ax_c  = fig.add_subplot(gs[0, 0], projection=proj)
ax_f  = fig.add_subplot(gs[0, 1], projection=proj)
ax_cb = fig.add_subplot(gs[0, 2])

for ax, label in [(ax_c, "ERA5 0.25\u00b0"),
                   (ax_f, "ERA5-Land 0.1\u00b0")]:
    ax.add_feature(BORDERS_10M)
    ax.add_feature(COAST_10M)
    ax.add_feature(LAND_10M)
    ax.set_extent(list(FULL_EXTENT), crs=proj)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, linestyle='--', color='gray')
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = {'size': 7}; gl.ylabel_style = {'size': 7}
    ax.set_title(label, fontsize=10, pad=4)
    add_map_labels(ax, transform=proj, fontsize=6.0, marker_size=4)

def lats_lons_vals(da):
    la = da.latitude.values; lo = da.longitude.values
    if la[0] > la[-1]:
        da = da.isel(latitude=slice(None, None, -1)); la = da.latitude.values
    return lo, la, da.values

lo_c, la_c, v_c = lats_lons_vals(tp_coarse_anim.isel(time=0))
lo_f, la_f, v_f = lats_lons_vals(tp_fine_anim.isel(time=0))

mesh_c = ax_c.pcolormesh(lo_c, la_c, v_c, cmap=PRECIP_CMAP, norm=PRECIP_NORM,
                          transform=proj, shading='auto')
mesh_f = ax_f.pcolormesh(lo_f, la_f, v_f, cmap=PRECIP_CMAP, norm=PRECIP_NORM,
                          transform=proj, shading='auto')

plot_peak_star(ax_f, best["Fine lon"], best["Fine lat"], transform=proj, ms=14)

sm = mpl.cm.ScalarMappable(cmap=PRECIP_CMAP, norm=PRECIP_NORM)
sm.set_array([])
cbar = fig.colorbar(sm, cax=ax_cb, extend='both')
cbar.set_label('Rainfall (mm/hr)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

time_text  = fig.text(0.5, 0.98, '', ha='center', va='top',
                       fontsize=12, fontweight='bold', transform=fig.transFigure)
coarse_ann = ax_c.text(0.02, 0.02, '', transform=ax_c.transAxes, fontsize=8, color='#222',
                        bbox=dict(facecolor='white', alpha=0.7, pad=2, edgecolor='none'))
fine_ann   = ax_f.text(0.02, 0.02, '', transform=ax_f.transAxes, fontsize=8, color='#222',
                        bbox=dict(facecolor='white', alpha=0.7, pad=2, edgecolor='none'))

def update(frame):
    snap_c = tp_coarse_anim.isel(time=frame)
    snap_f = tp_fine_anim.isel(time=frame)
    _, _, vc = lats_lons_vals(snap_c)
    _, _, vf = lats_lons_vals(snap_f)
    mesh_c.set_array(vc.ravel())
    mesh_f.set_array(vf.ravel())
    t_str = str(tp_fine_anim.time.values[frame])[:16].replace("T", " ") + " UTC"
    time_text.set_text(t_str)
    coarse_ann.set_text(f"Max: {float(snap_c.max()):.1f} mm/hr")
    fine_ann.set_text(  f"Max: {float(snap_f.max()):.1f} mm/hr")
    return mesh_c, mesh_f, time_text, coarse_ann, fine_ann

ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=400, blit=False)
gif_path = str(PROCESSED_OUTPUT / f"extreme_event_animation_{anim_year}.gif")
ani.save(gif_path, writer='pillow', fps=2.5, dpi=110)
print(f"\u2705 Animation saved: {gif_path}")
plt.close()


**Animation — 24-hour evolution of the largest extreme event, land-masked ERA5 0.25° (left) vs. ERA5-Land 0.1° (right).** Star marks the domain peak pixel; frame timestamp and running maxima are overlaid per frame. Not reproduced as a static figure — see the saved GIF.


### 7.9 Top-4 Events Panel

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7.9  TOP-4 EVENTS PANEL — 2x4 figure for paper/poster  (standalone)
#       Ported from corrdiff_interpolate_updated_v5.ipynb Section 10.
#       Uses the shared PRECIP_CMAP and the SHOW_PEAK_STAR toggle from viz_utils.
# ══════════════════════════════════════════════════════════════
from viz_utils import *

df_extreme = find_extreme_events()
tp_coarse_mm, _, _ = load_coarse_tp_masked()

top4 = df_extreme.head(4)
proj = ccrs.PlateCarree()
fig  = plt.figure(figsize=(18, 11))
gs   = gridspec.GridSpec(2, 4, figure=fig, hspace=0.08, wspace=0.05, top=0.93)

axes_top    = [fig.add_subplot(gs[0, c], projection=proj) for c in range(4)]
axes_bottom = [fig.add_subplot(gs[1, c], projection=proj) for c in range(4)]

for idx, row in enumerate(top4.itertuples()):
    year = int(row.Year)
    t_dt = pd.Timestamp(getattr(row, "_2"))
    peak_lat = getattr(row, "_3")   # 'Fine lat' -- note: the original v5
    peak_lon = getattr(row, "_4")   # 'Fine lon' -- ported cell used _5/_4
                                    # here, which actually plotted the
                                    # rainfall VALUE as a longitude. Fixed.

    fn   = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{year}.nc"
    ds_y = xr.open_dataset(fn)
    fine_snap = np.expm1(ds_y["tp"].sel(time=t_dt, method="nearest")).compute()
    ds_y.close()

    coarse_snap = tp_coarse_mm.sel(time=t_dt, method="nearest").sel(
        latitude=slice(LAT_MAX + 2, LAT_MIN - 2),
        longitude=slice(LON_MIN - 2, LON_MAX + 2)
    ).compute()

    def plot_panel(ax, data, title):
        la = data.latitude.values; lo = data.longitude.values
        if la[0] > la[-1]:
            data = data.isel(latitude=slice(None, None, -1)); la = data.latitude.values
        im = ax.pcolormesh(lo, la, data.values, cmap=PRECIP_CMAP, norm=PRECIP_NORM,
                           transform=proj, shading='auto')
        ax.add_feature(BORDERS_10M)
        ax.add_feature(COAST_10M)
        ax.set_extent(list(FULL_EXTENT), crs=proj)
        add_gridlines(ax, labels=False)
        ax.set_title(title, fontsize=12)
        add_map_labels(ax, transform=proj, fontsize=5.5, marker_size=4)
        return im

    im = plot_panel(axes_top[idx], coarse_snap, str(year))
    im = plot_panel(axes_bottom[idx], fine_snap, "")
    for ax in [axes_top[idx], axes_bottom[idx]]:
        plot_peak_star(ax, peak_lon, peak_lat, transform=proj, color='white', ms=10)

plt.colorbar(im, ax=axes_bottom, orientation='horizontal', fraction=0.025, shrink=0.8, pad=0.05,
             label='Hourly rainfall (mm/hr)', extend='both')

fig.text(0.08, 0.70, 'ERA5\n0.25\u00b0', ha='center', va='center', fontsize=12, fontweight='bold')
fig.text(0.08, 0.30, 'ERA5-Land\n0.1\u00b0', ha='center', va='center', fontsize=12, fontweight='bold')

mean_ratio = df_extreme["Ratio fine/coarse-max"].mean()  # reported in the caption, not on-figure
plt.savefig(str(PROCESSED_OUTPUT / "stat_top4_events_panel.png"), dpi=150,
            bbox_inches='tight', pad_inches=0.05)
plt.show()
print("\u2705 Top-4 panel saved.")


**Figure — Top-4 extreme hourly rainfall events over Vietnam, 2017–2025.** Columns are labelled by year; rows show land-masked ERA5 0.25° (top) vs. ERA5-Land 0.1° (bottom) for the same event. Stars mark the domain peak pixel; mean fine/coarse ratio across the 4 events is printed above (`mean_ratio`).


### 7.10 3-Panel Verification Plot (ERA5-Land TP pipeline check)

In [ ]:
from viz_utils import *

# ══════════════════════════════════════════════════════════════
# 7.10  3-PANEL VERIFICATION PLOT  (standalone)
#      Centred on the peak hourly rainfall event in a given month.
#      Panel 1: raw accumulated ERA5-Land TP (daily reset markers)
#      Panel 2: processed hourly ERA5-Land TP (bar chart)
#      Panel 3: piecewise cumulative — processed vs raw delta (mass-balance check)
# ══════════════════════════════════════════════════════════════
def visual_verification(check_year=2020, check_month=8, n_days=5):
    half_before = n_days // 2

    proc_file = PROCESSED_OUTPUT / f"era5_land_tp_vietnam_{check_year}.nc"
    if not proc_file.exists():
        print(f"  ⚠️  Processed file missing: {proc_file}")
        return

    # Find peak pixel in this month
    ds_proc     = xr.open_dataset(proc_file)
    month_start = f"{check_year}-{check_month:02d}-01"
    month_end   = (pd.Timestamp(month_start) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
    tp_month    = ds_proc["tp"].sel(time=slice(month_start, month_end))
    tp_month_mm = np.expm1(tp_month.values.astype(np.float64))

    flat_peak       = int(np.nanargmax(tp_month_mm))
    nt, nlat_p, nlon_p = tp_month_mm.shape
    ti_peak  = flat_peak // (nlat_p * nlon_p)
    rem      = flat_peak %  (nlat_p * nlon_p)
    lai_peak = rem // nlon_p
    loi_peak = rem %  nlon_p

    peak_time = pd.Timestamp(tp_month["time"].values[ti_peak])
    lat_peak  = float(tp_month["latitude"].values[lai_peak])
    lon_peak  = float(tp_month["longitude"].values[loi_peak])
    peak_val  = float(tp_month_mm[ti_peak, lai_peak, loi_peak])

    print(f"  Peak : {peak_val:.2f} mm/hr @ {peak_time}  lat={lat_peak:.2f}  lon={lon_peak:.2f}")

    win_start = pd.Timestamp(peak_time.date()) - pd.Timedelta(days=half_before)
    win_end   = win_start + pd.Timedelta(days=n_days)

    tp_proc_log = ds_proc["tp"].sel(time=slice(str(win_start), str(win_end - pd.Timedelta(hours=1))))
    tp_proc_mm  = np.expm1(
        tp_proc_log.sel(latitude=lat_peak, longitude=lon_peak, method="nearest")
        .values.astype(np.float64)
    )
    times_proc = tp_proc_log["time"].values
    ds_proc.close()

    # Load raw monthly files for the window
    raw_win_start = win_start - pd.Timedelta(hours=1)
    months_needed = set()
    cur = pd.Timestamp(raw_win_start.year, raw_win_start.month, 1)
    while cur <= win_end:
        months_needed.add((cur.year, cur.month))
        cur += pd.offsets.MonthBegin(1)

    raw_times_list, raw_vals_list = [], []
    for yr, mo in sorted(months_needed):
        f = RAW_OUTPUT_DIR / f"era5_land_tp_{yr}_{mo:02d}.nc"
        if not f.exists(): continue
        ds = xr.open_dataset(f)
        if "valid_time" in ds.dims: ds = ds.rename({"valid_time": "time"})
        ds = crop(ds)
        t  = ds["tp"]["time"].values
        v  = ds["tp"].sel(latitude=lat_peak, longitude=lon_peak, method="nearest").values.astype(np.float64) * 1000.0
        ds.close()
        raw_times_list.append(t); raw_vals_list.append(v)

    if not raw_times_list:
        print("  ⚠️  Raw files not found for window.")
        return

    raw_times_all = np.concatenate(raw_times_list)
    raw_vals_all  = np.concatenate(raw_vals_list)

    plot_mask      = (raw_times_all >= np.datetime64(win_start)) &                      (raw_times_all <= np.datetime64(win_end - pd.Timedelta(hours=1)))
    times_raw_plot = raw_times_all[plot_mask]
    vals_raw_plot  = raw_vals_all[plot_mask]

    # Panel 3 arrays
    run_base   = 0.0
    raw_zeroed = np.empty_like(vals_raw_plot)
    for i in range(len(times_raw_plot)):
        if pd.Timestamp(times_raw_plot[i]).hour == 1:
            run_base = vals_raw_plot[i]
        raw_zeroed[i] = vals_raw_plot[i] - run_base

    proc_hours_arr   = np.array([pd.Timestamp(t).hour for t in times_proc])
    cumsum_piecewise = np.zeros_like(tp_proc_mm)
    running = 0.0
    for i in range(len(tp_proc_mm)):
        if proc_hours_arr[i] == 1 and i > 0:
            running = 0.0
        running += tp_proc_mm[i]
        cumsum_piecewise[i] = running

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    def mark_resets(ax):
        for t in times_raw_plot:
            if pd.Timestamp(t).hour == 1:
                ax.axvline(pd.Timestamp(t), color="red", alpha=0.3, linewidth=0.8, linestyle="--")

    def mark_peak(ax):
        ax.axvline(peak_time, color="purple", alpha=0.6, linewidth=1.2, linestyle="-",
                   label=f"Peak {peak_val:.1f} mm/hr")

    ax1.plot(pd.DatetimeIndex(times_raw_plot), vals_raw_plot, color="steelblue",
             linewidth=1.2, label="Raw accumulated TP (mm)")
    mark_resets(ax1); mark_peak(ax1)
    ax1.axvline(pd.Timestamp(times_raw_plot[0]), color="red", alpha=0.3,
                linewidth=0.8, linestyle="--", label="01:00 UTC reset")
    ax1.set_ylabel("Accumulated TP (mm)"); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

    ax2.bar(pd.DatetimeIndex(times_proc), tp_proc_mm, width=pd.Timedelta(hours=1),
            color="darkorange", alpha=0.8, label="Processed hourly TP (mm/hr)")
    mark_resets(ax2); mark_peak(ax2)
    ax2.set_ylabel("Hourly TP (mm/hr)"); ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

    ax3.plot(pd.DatetimeIndex(times_proc), cumsum_piecewise, color="darkgreen",
             linewidth=1.4, label="Cumulative processed (mm, per run)")
    raw_plot_len = min(len(times_raw_plot), len(times_proc))
    ax3.plot(pd.DatetimeIndex(times_raw_plot[:raw_plot_len]), raw_zeroed[:raw_plot_len],
             color="steelblue", linewidth=1.0, linestyle=":",
             label="Raw accumulated (mm, zero-based per run)")
    mark_resets(ax3); mark_peak(ax3)
    ax3.set_ylabel("Piecewise cumulative TP (mm)"); ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.xticks(rotation=30, ha="right")

    plt.tight_layout()
    out_png = PROCESSED_OUTPUT / (
        f"tp_verification_{check_year}_{check_month:02d}"
        f"_peak{peak_time.strftime('%m%d_%H')}h.png"
    )
    plt.savefig(str(out_png), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✅  Plot saved → {out_png}")


# Run for August 2020 (typically a strong typhoon season month)
visual_verification(check_year=2024, check_month=5, n_days=5)

**Figure — ERA5-Land TP pipeline mass-balance check.** Top: raw accumulated TP with daily (01:00 UTC) reset markers; middle: processed hourly TP; bottom: piecewise-cumulative processed vs. zero-based raw accumulation, confirming the hourly-differencing step conserves mass. Location, month, and peak event are printed above (`lat_peak`, `lon_peak`, `peak_time`, `peak_val`).


### 7.11 Split Summary & Recommendation

In [ ]:
from viz_utils import *

print("Computing annual statistics (ERA5-Land 0.1°)...")
df_annual = compute_annual_stats()

# ══════════════════════════════════════════════════════════════
# 7.11  SPLIT SUMMARY TABLE  (standalone)
# ══════════════════════════════════════════════════════════════
print("=" * 70)
print("DATASET SPLIT SUMMARY")
print("=" * 70)

splits = {
    "TRAIN (2017–2023)": TRAIN_YEARS,
    "VAL   (2024)"     : VAL_YEARS,
    "TEST  (2025)"     : TEST_YEARS,
}

split_stats = {}
for split_name, years in splits.items():
    rows = df_annual[df_annual["Year"].isin(years)]
    print(f"\n{split_name}")
    print(f"  Years           : {years}")
    print(f"  Total hours     : {rows['Hours'].sum():,}")
    print(f"  Mean (mm/hr)    : {rows['Mean (mm/hr)'].mean():.4f}")
    print(f"  Max ever (mm/hr): {rows['Max (mm/hr)'].max():.2f}")
    print(f"  p99 range       : {rows['p99 (mm/hr)'].min():.2f} – {rows['p99 (mm/hr)'].max():.2f}")
    print(f"  Wet fraction    : {rows['Wet fraction'].mean():.4f}")
    split_stats[split_name] = rows

train_rows   = split_stats["TRAIN (2017–2023)"]
val_rows     = split_stats["VAL   (2024)"]
train_max    = train_rows["Max (mm/hr)"].max()
train_max_yr = int(train_rows.loc[train_rows["Max (mm/hr)"].idxmax(), "Year"])
val_max      = float(val_rows["Max (mm/hr)"].max())

print("\n\nRECOMMENDATION:")
print("  ✅  2017–2023 train | 2024 val | 2025 test is a solid chronological split.")
print(f"  ✅  Training-set peak is {train_max:.1f} mm/hr ({train_max_yr}) — "
      f"sufficient tail coverage for learning heavy-tail precipitation.")
print(f"  ✅  2024 validation captures a recent year with {val_max:.1f} mm/hr max.")
print("  ✅  2025 test is fully held-out; no temporal leakage possible.")
print("  ⚠️   The split is NOT random (required for time series) — do not shuffle.")